<a href="https://colab.research.google.com/github/Sajjad5/esg-risk-classification-finbert/blob/main/adv_ML_esg_risk_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Install Libraries

In [11]:
#!pip install transformers
#!pip install datasets
#!pip install accelerate
#!pip install evaluate
#!pip install scikit-learn

#2. Import Libraries

In [12]:
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# 3. Load Dataset

In [13]:
df = pd.read_csv('/content/all-data.csv', encoding='latin1')

df.head()

,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing ."
0,neutral,Technopolis plans to develop in stages an area...
1,negative,The international electronic industry company ...
2,positive,With the new production plant the company woul...
3,positive,According to the company 's updated strategy f...
4,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...


# 4. Preprocess Data

In [14]:
df = df.iloc[:, :2] # Select only the first two columns
df.columns = ['sentiment', 'text'] # Now, assign the desired column names

sentiment_map = {'neutral': 0, 'negative': 1, 'positive': 2}
df['sentiment_id'] = df['sentiment'].map(sentiment_map)

df.head()

,sentiment,text,sentiment_id
0,neutral,Technopolis plans to develop in stages an area...,0
1,negative,The international electronic industry company ...,1
2,positive,With the new production plant the company woul...,2
3,positive,According to the company 's updated strategy f...,2
4,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,2


# 5. Split Dataset

In [15]:
X = df['text']
y = df['sentiment_id']

# Split into training and temporary sets (80% train, 20% temp)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Split temporary set into validation and test sets (50% val, 50% test from temp)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f"Train data size: {X_train.shape[0]} samples")
print(f"Validation data size: {X_val.shape[0]} samples")
print(f"Test data size: {X_test.shape[0]} samples")

Train data size: 3876 samples
Validation data size: 484 samples
Test data size: 485 samples


# 6. Tokenize Data

In [16]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Create Dataset objects
train_dataset = Dataset.from_dict({'text': X_train.tolist(), 'labels': y_train.tolist()})
val_dataset = Dataset.from_dict({'text': X_val.tolist(), 'labels': y_val.tolist()})
test_dataset = Dataset.from_dict({'text': X_test.tolist(), 'labels': y_test.tolist()})

# Apply tokenization
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Example of tokenized training data:")
print(tokenized_train_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/484 [00:00<?, ? examples/s]

Map:   0%|          | 0/485 [00:00<?, ? examples/s]

Example of tokenized training data:
{'text': 'The buyer is real estate owner Propertos Oy , but the companies have agreed not to disclose financial details of the deal .', 'labels': 0, 'input_ids': [101, 1996, 17634, 2003, 2613, 3776, 3954, 5372, 13122, 1051, 2100, 1010, 2021, 1996, 3316, 2031, 3530, 2025, 2000, 26056, 3361, 4751, 1997, 1996, 3066, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,